# Fine-tuning RoBERTa
This notebook contains the shared preprocessing steps needed for RoBERTa plus the RoBERTa training, evaluation, and token-export cells. The BERT-specific cells are intentionally omitted.

In [1]:
import pandas as pd

# Load the dataset
dataset_path = "WELFake_Dataset.csv"
data = pd.read_csv(dataset_path)

# Remove duplicates
data = data.drop_duplicates()

# Display the number of rows and the first few rows of the dataset
print(f"Number of rows after removing duplicates: {len(data)}")
data.head()

Number of rows after removing duplicates: 72134


,Unnamed: 0,title,text,label
0,0,LAW ENFORCEMENT ON HIGH ALERT Following Threat...,No comment is expected from Barack Obama Membe...,1
1,1,NaN,Did they post their votes for Hillary already?,1
2,2,UNBELIEVABLE! OBAMA’S ATTORNEY GENERAL SAYS MO...,"Now, most of the demonstrators gathered last ...",1
3,3,"Bobby Jindal, raised Hindu, uses story of Chri...",A dozen politically active pastors came here f...,0
4,4,SATAN 2: Russia unvelis an image of its terrif...,"The RS-28 Sarmat missile, dubbed Satan 2, will...",1


# Fine-tuning RoBERTa
This section keeps only the RoBERTa pipeline: tokenization, dataset preparation, training, evaluation, and token export.

In [5]:
import torch
import torch.utils._pytree as _torch_pytree

# Compatibility shim for older torch builds used by transformers imports
def _register_pytree_node_compat(typ, flatten_fn, unflatten_fn, *args, **kwargs):
    return _torch_pytree._register_pytree_node(typ, flatten_fn, unflatten_fn)

_torch_pytree.register_pytree_node = _register_pytree_node_compat

from transformers import RobertaTokenizer, RobertaForSequenceClassification
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_recall_fscore_support, roc_auc_score
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from tqdm.auto import tqdm

# Reproducibility and device setup
torch.manual_seed(42)
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")

# Split the dataset into train and test (80/20)
data['label'] = data['label'].astype(int)
train_texts, test_texts, train_labels, test_labels = train_test_split(
    data['text'].astype(str),
    data['label'],
    test_size=0.2,
    random_state=42,
    stratify=data['label']
)

def tokenize_data(tokenizer, texts, max_length=256):
    return tokenizer(
        texts.tolist(),
        padding=True,
        truncation=True,
        max_length=max_length,
        return_tensors="pt"
    )

# Tokenizer
roberta_tokenizer = RobertaTokenizer.from_pretrained("roberta-base")

# Tokenize datasets
train_encodings_roberta = tokenize_data(roberta_tokenizer, train_texts)
test_encodings_roberta = tokenize_data(roberta_tokenizer, test_texts)

train_labels_tensor = torch.tensor(train_labels.values, dtype=torch.long)
test_labels_tensor = torch.tensor(test_labels.values, dtype=torch.long)

class FakeNewsDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: tensor[idx] for key, tensor in self.encodings.items()}
        item['labels'] = self.labels[idx]
        return item

train_dataset_roberta = FakeNewsDataset(train_encodings_roberta, train_labels_tensor)
test_dataset_roberta = FakeNewsDataset(test_encodings_roberta, test_labels_tensor)

batch_size = 8
train_loader_roberta = DataLoader(train_dataset_roberta, batch_size=batch_size, shuffle=True)
test_loader_roberta = DataLoader(test_dataset_roberta, batch_size=batch_size)

Using device: mps


In [8]:
def train_epoch(model, dataloader, optimizer, device):
    model.train()
    total_loss = 0.0

    for batch in tqdm(dataloader, desc="Training", leave=True, unit="batch"):
        batch = {k: v.to(device) for k, v in batch.items()}
        optimizer.zero_grad()
        outputs = model(**batch)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    return total_loss / len(dataloader)

@torch.no_grad()
def evaluate_model(model, dataloader, device, desc="Evaluating"):
    model.eval()
    total_loss = 0.0
    preds = []
    labels = []

    for batch in tqdm(dataloader, desc=desc, leave=True, unit="batch"):
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        total_loss += outputs.loss.item()

        batch_preds = torch.argmax(outputs.logits, dim=1)
        preds.extend(batch_preds.cpu().numpy())
        labels.extend(batch['labels'].cpu().numpy())

    return preds, labels, total_loss / len(dataloader)

In [11]:
import time
import os

# Fine-tune RoBERTa with pure PyTorch loop
roberta_model = RobertaForSequenceClassification.from_pretrained("roberta-base", num_labels=2).to(device)
optimizer_roberta = AdamW(roberta_model.parameters(), lr=2e-5, weight_decay=0.01)
num_epochs = 3
save_dir = "/Users/bahaar/Desktop/NLP/roberta_welfake_model"

epoch_bar = tqdm(range(num_epochs), desc="RoBERTa epochs", unit="epoch")
print("RoBERTa Classifier Head Details:")
print(roberta_model.classifier)
print("\nStarting RoBERTa fine-tuning...")

start_time = time.time()
for epoch in epoch_bar:
    epoch_start = time.time()
    train_loss = train_epoch(roberta_model, train_loader_roberta, optimizer_roberta, device)
    _, _, val_loss = evaluate_model(roberta_model, test_loader_roberta, device, desc=f"Validation {epoch + 1}/{num_epochs}")
    epoch_time = time.time() - epoch_start
    epoch_bar.set_postfix(train_loss=f"{train_loss:.4f}", val_loss=f"{val_loss:.4f}", sec=f"{epoch_time:.1f}")
    print(f"Epoch {epoch + 1}/{num_epochs} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Time: {epoch_time:.2f} sec")

os.makedirs(save_dir, exist_ok=True)
roberta_model.save_pretrained(save_dir)
roberta_tokenizer.save_pretrained(save_dir)

end_time = time.time()
print(f"RoBERTa fine-tuning completed in {end_time - start_time:.2f} seconds.")
print(f"Saved RoBERTa model and tokenizer to: {save_dir}")

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


RoBERTa epochs:   0%|          | 0/3 [00:00<?, ?epoch/s]

RoBERTa Classifier Head Details:
RobertaClassificationHead(
  (dense): Linear(in_features=768, out_features=768, bias=True)
  (dropout): Dropout(p=0.1, inplace=False)
  (out_proj): Linear(in_features=768, out_features=2, bias=True)
)

Starting RoBERTa fine-tuning...


Training:   0%|          | 0/7214 [00:00<?, ?batch/s]

...

Validation 2/3:   0%|          | 0/1804 [00:00<?, ?batch/s]

Epoch 2/3 | Train Loss: 0.0159 | Val Loss: 0.0211 | Time: 3920.27 sec


Training:   0%|          | 0/7214 [00:00<?, ?batch/s]

Validation 3/3:   0%|          | 0/1804 [00:00<?, ?batch/s]

Epoch 3/3 | Train Loss: 0.0249 | Val Loss: 0.0330 | Time: 3853.67 sec
RoBERTa fine-tuning completed in 11468.49 seconds.
Saved RoBERTa model and tokenizer to: /Users/bahaar/Desktop/NLP/roberta_welfake_model


In [12]:
import time
import os

if "roberta_model" not in globals() or "roberta_tokenizer" not in globals():
    candidate_model_dirs = [
        "/Users/bahaar/Desktop/NLP/roberta_welfake_model",
        "roberta_welfake_model",
    ]
    model_dir = next((path for path in candidate_model_dirs if os.path.isdir(path)), None)
    if model_dir is None:
        print("RoBERTa model not found. Run the training cell first or save the model to /Users/bahaar/Desktop/NLP/roberta_welfake_model.")
        model_dir = None
    else:
        from transformers import RobertaTokenizer, RobertaForSequenceClassification
        roberta_tokenizer = RobertaTokenizer.from_pretrained(model_dir)
        roberta_model = RobertaForSequenceClassification.from_pretrained(model_dir).to(device)
        print(f"Loaded RoBERTa model from: {model_dir}")

if "roberta_model" in globals() and "roberta_tokenizer" in globals():
    start_time = time.time()
    roberta_preds, roberta_true, roberta_eval_loss = evaluate_model(roberta_model, test_loader_roberta, device, desc="Final RoBERTa test")
    print(f"RoBERTa Test Loss: {roberta_eval_loss:.4f}")
    roberta_report = classification_report(roberta_true, roberta_preds)
    print("\nRoBERTa Classification Report:\n", roberta_report)
    print(f"RoBERTa test evaluation completed in {time.time() - start_time:.2f} seconds.")

Final RoBERTa test:   0%|          | 0/1804 [00:00<?, ?batch/s]

RoBERTa Test Loss: 0.0330

RoBERTa Classification Report:
               precision    recall  f1-score   support

           0       0.99      1.00      0.99      7006
           1       1.00      0.99      0.99      7421

    accuracy                           0.99     14427
   macro avg       0.99      0.99      0.99     14427
weighted avg       0.99      0.99      0.99     14427

RoBERTa test evaluation completed in 261.93 seconds.


In [13]:
# Add RoBERTa tokens to the dataset
data['roberta_tokens'] = data['text'].astype(str).apply(lambda x: roberta_tokenizer(x, padding=True, truncation=True, max_length=512, return_tensors="pt")['input_ids'].tolist())

# Save the updated dataset
data.to_csv("WELFake_Dataset_with_roberta_tokens.csv", index=False)

# External Dataset Evaluation
Here we test the trained RoBERTa model on a separate balanced dataset built from `True.csv` and `Fake.csv`.

In [15]:
import os
import pandas as pd
import torch
import time
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_recall_fscore_support,
    roc_auc_score,
)
from torch.utils.data import DataLoader, Dataset

# Paths provided by user
true_path = "/Users/bahaar/Desktop/NLP/True.csv"
fake_path = "/Users/bahaar/Desktop/NLP/Fake.csv"

# Load data
true_df = pd.read_csv(true_path)
fake_df = pd.read_csv(fake_path)

# Robust text extraction (prefers `text`, otherwise combines available text-like columns)
def build_text_series(df):
    if "text" in df.columns:
        return df["text"].astype(str)

    text_like_cols = [c for c in ["title", "text", "content", "article"] if c in df.columns]
    if text_like_cols:
        return df[text_like_cols].fillna("").astype(str).agg(" ".join, axis=1).str.strip()

    object_cols = df.select_dtypes(include=["object"]).columns.tolist()
    if not object_cols:
        raise ValueError("No usable text columns found in dataframe.")
    return df[object_cols].fillna("").astype(str).agg(" ".join, axis=1).str.strip()

true_texts = build_text_series(true_df)
fake_texts = build_text_series(fake_df)

# Balanced random sampling: 5000 from each class
n_per_class = 5000
if len(true_texts) < n_per_class or len(fake_texts) < n_per_class:
    raise ValueError(
        f"Not enough rows for balanced sampling. True={len(true_texts)}, Fake={len(fake_texts)}, required={n_per_class} each."
    )

true_sample = pd.DataFrame({"text": true_texts}).sample(n=n_per_class, random_state=42).copy()
true_sample["label"] = 0  # True news -> user label 0

fake_sample = pd.DataFrame({"text": fake_texts}).sample(n=n_per_class, random_state=42).copy()
fake_sample["label"] = 1  # Fake news -> user label 1

balanced_test_df = pd.concat([true_sample, fake_sample], ignore_index=True)
balanced_test_df = balanced_test_df.sample(frac=1, random_state=42).reset_index(drop=True)

print("Balanced external test set created.")
print("Class counts:")
print(balanced_test_df["label"].value_counts().sort_index())
print(f"Total rows: {len(balanced_test_df)}")

# Tokenize and build dataloader for RoBERTa
ext_texts = balanced_test_df["text"].astype(str)
ext_labels = torch.tensor(balanced_test_df["label"].values, dtype=torch.long)
ext_encodings = roberta_tokenizer(
    ext_texts.tolist(),
    padding=True,
    truncation=True,
    max_length=256,
    return_tensors="pt",
)

class ExternalEvalDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item["labels"] = self.labels[idx]
        return item

external_dataset = ExternalEvalDataset(ext_encodings, ext_labels)
external_loader = DataLoader(external_dataset, batch_size=16)

# Load a saved RoBERTa model if the current session does not already have one
if "roberta_model" not in globals() or "roberta_tokenizer" not in globals():
    candidate_model_dirs = [
        "/Users/bahaar/Desktop/NLP/roberta_welfake_model",
        "roberta_welfake_model",
    ]
    model_dir = next((path for path in candidate_model_dirs if os.path.isdir(path)), None)
    if model_dir is None:
        print("RoBERTa model not found. Run the training cell first or save the model to /Users/bahaar/Desktop/NLP/roberta_welfake_model.")
    else:
        from transformers import RobertaTokenizer, RobertaForSequenceClassification
        roberta_tokenizer = RobertaTokenizer.from_pretrained(model_dir)
        roberta_model = RobertaForSequenceClassification.from_pretrained(model_dir).to(device)
        print(f"Loaded RoBERTa model from: {model_dir}")

if "roberta_model" in globals() and "roberta_tokenizer" in globals():
    # Evaluate RoBERTa on external balanced set
    start_time = time.time()
    roberta_model.eval()
    all_preds, all_true, all_probs = [], [], []
    total_loss = 0.0

    with torch.no_grad():
        for batch in tqdm(external_loader, desc="External evaluation", leave=True, unit="batch"):
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = roberta_model(**batch)

            total_loss += outputs.loss.item()

            probs = torch.softmax(outputs.logits, dim=1)[:, 1]
            preds = torch.argmax(outputs.logits, dim=1)

            all_probs.extend(probs.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())
            all_true.extend(batch["labels"].cpu().numpy())

    avg_loss = total_loss / len(external_loader)
    accuracy = accuracy_score(all_true, all_preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        all_true, all_preds, average="binary", zero_division=0
    )
    auc = roc_auc_score(all_true, all_probs)
    cm = confusion_matrix(all_true, all_preds)
    tn, fp, fn, tp = cm.ravel()
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0

    print("\n===== RoBERTa Evaluation on External Balanced Test Set =====")
    print(f"Loss: {avg_loss:.4f}")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Precision (class 1 / Fake): {precision:.4f}")
    print(f"Recall (class 1 / Fake): {recall:.4f}")
    print(f"F1-score (class 1 / Fake): {f1:.4f}")
    print(f"Specificity (class 0 / True recall): {specificity:.4f}")
    print(f"ROC-AUC: {auc:.4f}")

    print("\nConfusion Matrix [[TN, FP], [FN, TP]]:")
    print(cm)

    print("\nDetailed Classification Report:")
    print(
        classification_report(
            all_true,
            all_preds,
            target_names=["True (0)", "Fake (1)"],
            digits=4,
            zero_division=0,
        )
    )

    # Diagnostic: check if the model's label mapping is opposite of your 0/1 convention
    flipped_preds = [1 - p for p in all_preds]
    flipped_probs = [1.0 - p for p in all_probs]

    acc_flip = accuracy_score(all_true, flipped_preds)
    prec_flip, rec_flip, f1_flip, _ = precision_recall_fscore_support(
        all_true, flipped_preds, average="binary", zero_division=0
    )
    auc_flip = roc_auc_score(all_true, flipped_probs)
    cm_flip = confusion_matrix(all_true, flipped_preds)

    print("===== Diagnostic Report with Flipped Prediction Labels =====")
    print(f"Accuracy: {acc_flip:.4f}")
    print(f"Precision (class 0 / True): {prec_flip:.4f}")
    print(f"Recall (class 0 / True): {rec_flip:.4f}")
    print(f"F1-score (class 0 / True): {f1_flip:.4f}")
    print(f"ROC-AUC: {auc_flip:.4f}")
    print("\nConfusion Matrix [[TN, FP], [FN, TP]]:")
    print(cm_flip)
    print("\nDetailed Classification Report:")
    print(
        classification_report(
            all_true,
            flipped_preds,
            target_names=["True (0)", "Fake (1)"],
            digits=4,
            zero_division=0,
        )
    )

    # Save sampled test set for reproducibility
    balanced_test_df.to_csv("/Users/bahaar/Desktop/NLP/balanced_external_test_10k.csv", index=False)
    print("Saved sampled test set to /Users/bahaar/Desktop/NLP/balanced_external_test_10k.csv")
    print(f"External evaluation completed in {time.time() - start_time:.2f} seconds.")

Balanced external test set created.
Class counts:
label
0    5000
1    5000
Name: count, dtype: int64
Total rows: 10000


External evaluation:   0%|          | 0/625 [00:00<?, ?batch/s]


===== RoBERTa Evaluation on External Balanced Test Set =====
Loss: 0.0081
Accuracy: 0.9982
Precision (class 1 / Fake): 0.9980
Recall (class 1 / Fake): 0.9984
F1-score (class 1 / Fake): 0.9982
Specificity (class 0 / True recall): 0.9980
ROC-AUC: 1.0000

Confusion Matrix [[TN, FP], [FN, TP]]:
[[4990   10]
 [   8 4992]]

Detailed Classification Report:
              precision    recall  f1-score   support

    True (0)     0.9984    0.9980    0.9982      5000
    Fake (1)     0.9980    0.9984    0.9982      5000

    accuracy                         0.9982     10000
   macro avg     0.9982    0.9982    0.9982     10000
weighted avg     0.9982    0.9982    0.9982     10000

===== Diagnostic Report with Flipped Prediction Labels =====
Accuracy: 0.0018
Precision (class 0 / True): 0.0016
Recall (class 0 / True): 0.0016
F1-score (class 0 / True): 0.0016
ROC-AUC: 0.0000

Confusion Matrix [[TN, FP], [FN, TP]]:
[[  10 4990]
 [4992    8]]

Detailed Classification Report:
              precision  

In [16]:
import os
import torch

save_dir = "/Users/bahaar/Desktop/NLP"
os.makedirs(save_dir, exist_ok=True)

# Move tensors to CPU for portable checkpoints
state_dict_cpu = {k: v.detach().cpu() for k, v in roberta_model.state_dict().items()}

pt_path = os.path.join(save_dir, "bert_welfake.pt")
pth_path = os.path.join(save_dir, "bert_welfake.pth")

# Use legacy (non-zip) serialization to avoid the torch zip-writer issue in this environment
torch.save(state_dict_cpu, pt_path, _use_new_zipfile_serialization=False)
torch.save(state_dict_cpu, pth_path, _use_new_zipfile_serialization=False)

print(f"Saved .pt checkpoint to: {pt_path}")
print(f"Saved .pth checkpoint to: {pth_path}")

Saved .pt checkpoint to: /Users/bahaar/Desktop/NLP/bert_welfake.pt
Saved .pth checkpoint to: /Users/bahaar/Desktop/NLP/bert_welfake.pth
